Evan Edelstein
EN.605.645.82.SP26

# Module 8 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [1]:
import json
import random
from copy import deepcopy
from math import inf, log2
from typing import Dict, List, NamedTuple, Tuple, Callable

## Decision Trees

For this assignment you will be implementing and evaluating a Decision Tree using the ID3 Algorithm (**no** pruning or normalized information gain). Use the provided pseudocode. The data is located at (copy link):

http://archive.ics.uci.edu/ml/datasets/Mushroom

**Just in case** the UCI repository is down, which happens from time to time, I have included the data and name files on Canvas.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can used Dicts, NamedTuples, etc. as your abstract data type (ADT) for the the tree and nodes.
    </p>
</div>

One of the things we did not talk about in the lectures was how to deal with missing values. There are two aspects of the problem here. What do we do with missing values in the training data? What do we do with missing values when doing classifcation?

There are a lot of different ways that we can handle this.
A common algorithm is to use something like kNN to impute the missing values.
We can use conditional probability as well.
There are also clever modifications to the Decision Tree algorithm itself that one can make.

We're going to do something simpler, given the size of the data set: remove the observations with missing values ("?").

You must implement the following functions:

`train` takes training_data and returns the Decision Tree as a data structure.

```
def train(training_data):
   # returns the Decision Tree.
```

`classify` takes a tree produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data).

```
def classify(tree, observations):
    # returns a list of classifications
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application).

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**

```
def pretty_print_tree(tree):
    # pretty prints the tree
```

This should be a text representation of a decision tree trained on the entire data set (no train/test).

To summarize...

Apply the Decision Tree algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. When you are done, apply the Decision Tree algorithm to the entire data set and print out the resulting tree.

**Note** Because this assignment has a natural recursive implementation, you should consider using `deepcopy` at the appropriate places.


### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [2]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [3]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

# Tree

In [4]:
Node = NamedTuple("Node", [("node", str), ("children", List)])

In [5]:
def create_node(attribute: str) -> Node:
    return Node(attribute, [])

In [6]:
node = create_node("foo")
assert node.node == "foo"  # test 1 - has node
assert node.children == []  # test 2 - has children

node = create_node("")
assert node.node == ""  # test 3 empty list

In [7]:
def add_child(parent: Node, child: Node, value: str) -> Node:
    if any(value == v for v, _ in parent.children):
        return parent
    parent.children.append((value, child))
    return parent

In [8]:
parent = create_node("child")
child1 = create_node("child1")
child2 = create_node("child2")

parent = add_child(parent, child1, "edge1")
assert parent.children == [("edge1", child1)]  # test 1 - add child node and edge to tree

parent = add_child(parent, child2, "edge2")
assert parent.children == [("edge1", child1), ("edge2", child2)]  # test 2 - add child node and edge to children

parent = add_child(parent, child1, "edge2")
assert parent.children == [("edge1", child1), ("edge2", child2)]  # test 3 - dont add child if already in edge already taken

In [9]:
def pretty_print_tree(node: Node, indent: int = 0):
    tabs = " " * indent
    print(f"{tabs} - {node.node}")  # parent (attr to be split)
    for attr, child in node.children:
        if not child.children:
            print(f"{tabs} | {attr} -> {child.node}")  # leaf node
        else:
            print(f"{tabs} | {attr}")  # internal node
            pretty_print_tree(child, indent + 8)
    return

In [10]:
tree = Node(node="a", children=[("1", Node(node="b", children=[]))])
pretty_print_tree(tree)  # test 1 - one child leaf


print("\n\n")

tree = Node("", [])
pretty_print_tree(tree)  # test 2 - empty list

print("\n\n")

tree = Node(
    "1",
    [
        ("a", Node("y", [])),
        ("b", Node("2", [("c", Node("n", []))])),
    ],
)
pretty_print_tree(tree)  # test 3 - multiple children


 - a
 | 1 -> b



 - 



 - 1
 | a -> y
 | b
         - 2
         | c -> n


In [11]:
def traverse_tree(node: Node, observation: List[str], features: Dict[int, str]) -> str | None:
    if len(node.children) == 0:  # leaf
        return node.node
    
    attribute = node.node
    column = [k for k, v in features.items() if v == attribute]  # reverse lookup
    if len(column) == 0:
        return None
    
    observed_value = observation[column[0]]    
    for value, child in node.children:
        if value == observed_value:
            return traverse_tree(child, observation, features)
    return None

In [12]:
tree = Node("1", [("a", Node("y", [])), ("b", Node("n", []))])

assert traverse_tree(tree, ["a"], {0: "1"}) == "y"  # test 1 - observation in tree returns leaf label
assert traverse_tree(tree, ["c"], {0: "1"}) is None  # test 2 - observation not in tree return None

tree = Node("1", [("a", Node("2", [("b", Node("y", [])), ("c", Node("n", []))]))])
assert traverse_tree(tree, ["a", "b"], {0: "1", 1: "2"}) == "y"  # test 3 - search through tree

# I/O and Data Parsing

In [13]:
def parse_attributes(filename: str) -> Tuple[Dict[str, List[str]], Dict[str, Dict[str, str]]]:
    abrv2fullname: Dict[str, Dict[str, str]] = {}
    attributes: Dict[str, List[str]] = {}

    with open(filename, "r") as fh:
        data: Dict[str, Dict[str, str]] = json.load(fh)

    for feature, attr in data.items():
        abrv2fullname[feature] = {}
        attributes[feature] = []
        for single_letter, name in attr.items():
            attributes[feature].append(name)
            abrv2fullname[feature][single_letter] = name
            
    return attributes, abrv2fullname

In [14]:
filename = "./agaricus-lepiota-3.attrs"
attributes, abrv2fullname = parse_attributes(filename)

attribute_keys = [
    "mushroom-type",
    "cap-shape",
    "cap-surface",
    "cap-color",
    "bruises",
    "odor",
    "gill-attachment",
    "gill-spacing",
    "gill-size",
    "gill-color",
    "stalk-shape",
    "stalk-root",
    "stalk-surface-above-ring",
    "stalk-surface-below-ring",
    "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-type",
    "veil-color",
    "ring-number",
    "ring-type",
    "spore-print-color",
    "population",
    "habitat",
]

assert list(attributes.keys()) == attribute_keys  # test 1 - all keys are present
assert all([len(v) > 1 for v in a] for a in attributes.values())  # test 2 - all attribute names are full name
assert all([len(k) == 1 and len(v) > 0 for k, v in a.items()] for a in abrv2fullname.values())  # test 3 - can map from single letter to full name


In [15]:
def rename_data(data: List[List[str]], attributes: Dict[str, List[str]], abrv2fullname: Dict[str, Dict[str, str]]) -> List[List[str]] | None:
    new_data = []
    for row in data:
        if len(row) != len(attributes):
            return None

        new_row = []
        for value, attr in zip(row, attributes):
            if attr in abrv2fullname and value in abrv2fullname[attr]:
                new_row.append(abrv2fullname[attr][value])
            else:
                return None
        new_data.append(new_row)

    return new_data

In [16]:
data = [["a", "b", "c"], ["a", "b", "c"]]
attributes = {"1": ["a"], "2": ["b"], "3": ["c"]}
abrv2fullname = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2fullname)
assert result is not None and result[0][0] == "aaa" and result[0][1] == "bbb" and result[0][2] == "ccc" and result[1][0] == "aaa" and result[1][1] == "bbb" and result[1][2] == "ccc"  # test 1 - normal replacement


data = [["a", "b", "c"]]
attributes = {"1": ["a"], "2": ["b"], "3": ["c"]}
abrv2fullname = {"2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2fullname)
assert result is None  # test 2 - missing attribute in map


data = [["a", "b", "c", "d"]]
attributes = {"1": ["a"], "2": ["b"], "3": ["c"]}
abrv2fullname = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}
result = rename_data(data, attributes, abrv2fullname)
assert result is None  # test 3 - extra value in data

In [17]:
def split_features(attributes: Dict[str, List[str]], label: str) -> Tuple[Dict[int, str], List[str] | None, int]:
    features: Dict[int, str] = {}
    labels: List[str] | None = None
    label_index = -1
    for index, attr in enumerate(attributes):
        if attr == label:
            labels = attributes[attr]
            label_index = index
        else:
            features[index] = attr
    return features, labels, label_index

In [18]:
attributes = {"1": ["a"], "2": ["b"], "3": ["c"]}
label = "3"
features, labels, label_index = split_features(attributes, label)
assert features == {0: "1", 1: "2"} and labels == ["c"] and label_index == 2  # test 1 - splits features and labels

attributes = {"1": ["a"], "2": ["b"], "3": ["c"]}
label = "4"
features, labels, label_index = split_features(attributes, label)
assert features == {0: "1", 1: "2", 2: "3"} and labels is None and label_index == -1  # test 2 - label doesn't exist in attributes


attributes = {"1": ["a"]}
label = "1"
features, labels, label_index = split_features(attributes, label)
assert features == {} and labels == ["a"] and label_index == 0  # test 3 - only label so features is empty


In [19]:
def mask_label(data, label_index):
    return [[value for idx, value in enumerate(row) if idx != label_index] for row in data]

In [20]:
data = [["a", "b", "c"], ["a", "b", "c"]]
assert mask_label(data, 0) == [["b", "c"], ["b", "c"]]  # test 1 - mask off first column
assert mask_label(data, 4) == [["a", "b", "c"], ["a", "b", "c"]]  # test 2 - index out of range doesnt effect data
assert mask_label([], 4) == []  # test 3 - empty list

# ID3

In [21]:
def is_homogeneous(data: List[List[str]], label_index: int) -> bool:
    if len(data) and all(len(row) and 0 <= label_index < len(row) for row in data) and len(set(row[label_index] for row in data)) == 1:
        return True
    return False

In [22]:
data = [["a", "b", "c"], ["a", "b", "c"]]
assert is_homogeneous(data, 0) == True  # test 1 - column is the same

data = [["a", "b"], ["b", "c"]]
assert is_homogeneous(data, 1) == False  # test 2 - column is different

data = [["a", "b", "c"], ["a", "b", "c"]]
assert is_homogeneous(data, 4) == False  # test 3 - out of bounds label index

data = [[], []]
assert is_homogeneous(data, 0) == False  # test 3 - empty lists arent homogenous

In [23]:
def get_first_label(data: List[List[str]], label_index: int) -> str | None:
    if len(data) and 0 <= label_index < len(data[0]):
        return data[0][label_index]
    return None

In [24]:
data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_first_label(data, 0) == "a"  # test 1 - get first label

data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_first_label(data, 4) is None  # test 2 - out of bounds label index

data = [[]]
assert get_first_label(data, 4) is None  # test 3 - empty lists returns None


In [25]:
def get_majority_label(data: List[List[str]], label_index: int, trace: bool = False) -> str | None:
    if not data or not (0 <= label_index < len(data[0])):
        return None

    counts = {}
    for row in data:
        label = row[label_index]
        if label not in counts:
            counts[label] = 1
        else:
            counts[label] += 1

    label = max(counts, key=lambda k: counts[k])

    print(f"Majority label: {label}") if trace else None
    return label

In [26]:
data = [["a", "b", "c"], ["a", "b", "c"], ["c", "b", "a"]]
assert get_majority_label(data, 0) == "a"  # test 1 - get first label

data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_majority_label(data, 4) is None  # test 2 - out of bounds label index

data = [[]]
assert get_first_label(data, 0) is None  # test 3 - empty lists returns None

In [27]:
def count_values_by_label(data: List[List[str]], attr_index: int, attr: str, attributes: Dict[str, List[str]], label_index: int, labels: List[str]) -> Tuple[Dict[str, int], Dict[str, Dict[str, int]], int] | None:
    subset_sizes: Dict[str, int] = {i: 0 for i in attributes[attr]}
    label_counts: Dict[str, Dict[str, int]] = {l: {a: 0 for a in attributes[attr]} for l in labels}
    total_size = 0

    for row in data:
        if not (0 <= attr_index < len(row)) or not (0 <= label_index < len(row)):
            return None

        observation = row[attr_index]
        label = row[label_index]

        if label not in labels or observation not in attributes[attr]:
            return None

        subset_sizes[observation] += 1
        label_counts[label][observation] += 1
        total_size += 1
    return subset_sizes, label_counts, total_size

In [28]:
data = [["a", "b", "c", "y"], ["a", "b", "c", "y"], ["c", "b", "a", "n"]]
attributes = {"1": ["a", "b", "c"], "2": ["a", "b", "c"], "3": ["a", "b", "c"]}
label_idx = 3
labels = ["y", "n"]

result = count_values_by_label(data, 0, "1", attributes, label_idx, labels)
assert result is not None  # test 1 - result is not None

subset_sizes, label_counts, total_size = result
assert subset_sizes == {"a": 2, "b": 0, "c": 1} and label_counts == {"y": {"a": 2, "b": 0, "c": 0}, "n": {"a": 0, "b": 0, "c": 1}} and total_size == 3  # test 2 - result is correct

data = [["a", "b", "c", "y"], ["a", "b", "c", "y"], ["c", "b", "a", "n"]]
attributes = {"1": ["b", "c"], "2": ["a", "b", "c"], "3": ["a", "b", "c"]}
label_idx = 3
labels = ["y", "n"]

assert count_values_by_label(data, 0, "1", attributes, label_idx, labels) == None  # test 3 - missing attribute retunrs None

In [29]:
def calculate_entropy(data: List[List[str]], attr_index: int, attr: str, attributes: Dict[str, List[str]], label_index: int, labels: List[str]) -> float | None:
    result = count_values_by_label(data, attr_index, attr, attributes, label_index, labels)
    if result is None:
        return None
    subset_sizes, label_counts, total_size = result
    entropy = 0

    for feature in attributes[attr]:
        subset_size, subset_entropy = subset_sizes[feature], 0
        if subset_size == 0:
            continue

        for label in labels:
            p = label_counts[label][feature] / subset_size
            if p <= 0.0:
                continue
            subset_entropy += -1 * p * log2(p)
        entropy += (subset_size / total_size) * subset_entropy
    return entropy

In [30]:
data = [["a", "y"], ["a", "y"], ["b", "n"], ["b", "n"]]
attributes = {"1": ["a", "b"]}
assert calculate_entropy(data, 0, "1", attributes, 1, ["y", "n"]) == 0  # test 1 - perfect split has entropy of 0

data = [["a", "y"], ["b", "y"], ["a", "n"], ["b", "n"]]
assert calculate_entropy(data, 0, "1", attributes, 1, ["y", "n"]) == 1  # test 1 - 50/50 split has entropy of 1

data = [["a", "y"], ["b", "y"], ["a", "n"], ["b", "n"]]
attributes = {"1": ["a"]}
assert calculate_entropy(data, 0, "1", attributes, 1, ["y", "n"]) is None  # test 1 - missing attribute returns None

In [31]:
def pick_best_attribute(data: List[List[str]], features: Dict[int, str], attributes: Dict[str, List[str]], label_idx: int, labels: List[str], trace: bool = False) -> Tuple[int, str | None]:
    best_attr, best_entropy, best_attr_idx = None, inf, -1

    for attr_idx, attr in features.items():
        entropy = calculate_entropy(data, attr_idx, attr, attributes, label_idx, labels)
        if entropy is None:
            print(f"Entropy of index {attr_idx} - attr {attr} is None") if trace else None
            continue
        print(f"Entropy of index {attr_idx} - attr {attr} = {entropy:.3f}") if trace else None

        if entropy < best_entropy:
            best_attr = attr
            best_entropy = entropy
            best_attr_idx = attr_idx

    if trace:
        text = f"index {best_attr_idx} - attr {best_attr} = {best_entropy:.3f}" if best_attr else "not found"
        print(f"Lowest Entropy Attribute: {text}") 
    return best_attr_idx, best_attr

In [32]:
data = [["a", "c", "y"], ["a", "d", "y"], ["b", "c", "n"], ["b", "d", "n"]]
attributes = {"1": ["a", "b"], "2": ["c", "d"]}
features = {0: "1", 1: "2"}
idx, attr = pick_best_attribute(data, features, attributes, 2, ["y", "n"], trace=True)
assert idx == 0 and attr == "1"  # test 1 - first column splits data


data = [["a", "c", "n"], ["b", "c", "y"], ["b", "d", "n"], ["b", "d", "n"]]
attributes = {"1": ["a", "b"], "2": ["c", "d"]}
features = {0: "1", 1: "2"}
idx, attr = pick_best_attribute(data, features, attributes, 2, ["y", "n"], trace=True)
assert idx == 1 and attr == "2"  # test 2 - picks attr with lower entropy


data = [["a", "c", "y"], ["b", "d", "n"]]
attributes = {"1": ["a"], "2": ["c"]}
features = {0: "1", 1: "2"}
idx, attr = pick_best_attribute(data, features, attributes, 2, ["y", "n"], trace=True)
assert attr is None and idx == -1  # test 3 - no valid entropy when attributes are missing

Entropy of index 0 - attr 1 = 0.000
Entropy of index 1 - attr 2 = 1.000
Lowest Entropy Attribute: index 0 - attr 1 = 0.000
Entropy of index 0 - attr 1 = 0.689
Entropy of index 1 - attr 2 = 0.500
Lowest Entropy Attribute: index 1 - attr 2 = 0.500
Entropy of index 0 - attr 1 is None
Entropy of index 1 - attr 2 is None
Lowest Entropy Attribute: not found


In [33]:
def domain(attributes: Dict[str, List[str]], feature: str, nans: List[str] | None = None) -> List[str]:
    if nans is None:
        nans = ["?"]
    return [i for i in attributes[feature] if i not in nans]

In [34]:
attributes = {"1": ["a", "b", "c"]}
assert domain(attributes, "1") == ["a", "b", "c"]  # test 1 - normal

attributes = {"1": ["a", "b", "?"]}
assert domain(attributes, "1") == ["a", "b"]  # test 2 - remove missing


attributes = {"1": ["?", "na"]}
assert domain(attributes, "1", ["?", "na"]) == []  # test 3 - returns empty list if all attributes are missing

In [35]:
def subset_data(data: List[List[str]], attr_index: int, attr: str) -> List[List[str]]:
    return [deepcopy(row) for row in data if row[attr_index] == attr]

In [36]:
data = [["a", "y"], ["b", "y"], ["a", "n"]]
assert subset_data(data, 0, "a") == [["a", "y"], ["a", "n"]]  # test 1 - get rows

result = subset_data(data, 0, "a")
result[0][0] = "A"
assert data[0][0] == "a"  # test 2 - deepcopy

assert subset_data(data, 0, "c") == []  # test 3 - empty list if no match

In [37]:
def remove_features(features: Dict[int, str], attr_index: int) -> Dict[int, str]:
    new_features = deepcopy(features)
    new_features.pop(attr_index)
    return new_features

In [38]:
# remove_features
features = {0: "1", 1: "2", 2: "3"}
assert remove_features(features, 2) == {0: "1", 1: "2"}  # test 1 - remove feature

original = {0: "1", 1: "2"}
remove_features(original, 0)
assert original == {0: "1", 1: "2"}  # test 2 - deepcopy works

assert remove_features({0: "1"}, 0) == {}  # test 3 - remove last key returns empty dict

In [39]:
def base_cases(data: List[List[str]], features: Dict[int, str], label_index: int, default_label: str | None = None, trace: bool = False) -> Node | None:
    if len(data) == 0:
        print("Base Case - empty data") if trace else None
        majority_label = get_majority_label(data, label_index) if default_label is None else default_label
        return create_node(majority_label) if majority_label is not None else None

    if is_homogeneous(data, label_index):
        print("Base Case - homogenous data") if trace else None
        node = get_first_label(data, label_index)
        if node is None:
            raise Exception("Data set is homogenous and empty")
        return create_node(node)

    if len(features) == 0:
        print("Base Case - empty features") if trace else None
        majority_label = get_majority_label(data, label_index)
        return create_node(majority_label) if majority_label is not None else None
    return None

In [40]:
tree = base_cases([], {0: "1"}, 1, default_label="y")
assert tree is not None and tree.node == "y"  # test 1 - empty data uses default label

data = [["a", "n"], ["b", "n"]]
tree = base_cases(data, {0: "1"}, 1)
assert tree is not None and tree.node == "n"  # test 2 - homogeneous data

data = [["a", "y"], ["b", "y"], ["c", "n"]]
tree = base_cases(data, {}, 1)
assert tree is not None and tree.node == "y"  # test 3 - empty feature list returns majority

assert base_cases(data, {0: "1"}, 1) is None  # test 4 - no base cases hit return None

In [41]:
def id3(data: List[List[str]], features: Dict[int, str], attributes: Dict[str, List[str]], label_index: int, labels: List[str], default_label: str | None = None, trace: bool = False) -> Node:
    print(f"{features=}, {attributes=}") if trace else None

    result = base_cases(data, features, label_index, default_label, trace)
    if result is not None:
        return result

    index, attr = pick_best_attribute(data, features, attributes, label_index, labels)
    if attr is None:
        raise Exception("Cannot find a feature to partition")

    node = create_node(attr)
    default_label = get_majority_label(data, label_index, trace)

    for value in domain(attributes, attr):
        subset = subset_data(data, index, value)
        new_features = remove_features(features, index)
        child = id3(subset, new_features, attributes, label_index, labels, default_label, trace)
        node = add_child(node, child, value)
    return node

In [42]:
# data = [["a", "y"], ["b", "y"], ["a", "n"]]
# attributes = {"1": ["a", "b"], "label": ["y", "n"]}
# features = {0: "1"}
# tree = id3(data, features, attributes, 1, ["y", "n"])
# pretty_print_tree(tree)

# assert tree.node == "1"  # test 1 - root is "1"
# assert tree.children == ("a", Node("y", []))  # test 2 - children

# Model 

In [43]:
def train(data: List[List[str]], features: Dict[int, str], attributes: Dict[str, List[str]], label_idx: int, labels: List[str], trace=False) -> Node | None:
    if labels is None:
        return None
    tree = id3(data, features, attributes, label_idx, labels, None, trace)
    return tree

In [44]:
def classify(tree, observations, features):
    classifications = []
    for row in observations:
        label = traverse_tree(tree, row, features)
        classifications.append(label)
    return classifications

In [45]:
def evaluate(data, classifications, label_idx, labels, confusion_matrix):
    errors = 0

    for row, estimate in zip(data, classifications):
        if row[label_idx] == estimate and estimate == labels[1]:
            confusion_matrix["TP"] += 1
        elif row[label_idx] == estimate and estimate == labels[0]:
            confusion_matrix["TN"] += 1
        elif row[label_idx] != estimate and row[label_idx] == labels[0]:
            confusion_matrix["FP"] += 1
            errors += 1
        elif row[label_idx] != estimate and row[label_idx] == labels[1]:
            confusion_matrix["FN"] += 1
            errors += 1
        else:
            raise Exception(f"Unable to classify estimate: {estimate}")

    return errors, confusion_matrix

In [46]:
def cross_validate(data: List[List[str]], features: Dict[int, str], attributes: Dict[str, List[str]], label_idx, labels: List[str], train_fn: Callable = train, classify_fn: Callable = classify, eval_fn: Callable = evaluate, n_folds: int = 10, trace: bool = False):
    random.shuffle(data)
    folds = create_folds(data, n_folds)
    confusion_matrix: Dict[str, int] = {"TN": 0, "TP": 0, "FN": 0, "FP": 0}

    total_errors = 0
    for idx, test_fold in enumerate(folds):
        training_set = []
        for idx2, train_fold in enumerate(folds):
            if idx == idx2:
                continue
            training_set.extend(train_fold)

        tree = train_fn(training_set, features, attributes, label_idx, labels, trace)
        classifications = classify_fn(tree, test_fold, features)
        errors, confusion_matrix = eval_fn(test_fold, classifications, label_idx, labels, confusion_matrix)
        total_errors += errors
    error_rate = total_errors / len(data) if len(data) else 0.0
    return error_rate, confusion_matrix

# Run

In [47]:
def run_model(data, attributes, label):
    n_folds = 10

    features, labels, label_idx = split_features(attributes, label)
    assert labels is not None

    avrg_error_rate, cm = cross_validate(data, features, attributes, label_idx, labels, n_folds=n_folds)
    print(f"\nConfusion Matrix ({n_folds}-fold CV):")
    print(f"TP={cm['TP']}  FP={cm['FP']}\nFN={cm['FN']}  TN={cm['TN']}")
    print(f"\nAverage Error Rate ({n_folds}-fold CV): {avrg_error_rate:0.4f}")

    tree = train(data, features, attributes, label_idx, labels, trace=False)
    assert tree is not None

    print("\nDecision Tree:")
    pretty_print_tree(tree)

In [48]:
attributes = {"Shape": ["round", "square"], "Size": ["large", "small"], "Color": ["blue", "green", "red"], "Safe?": ["yes", "no"]}
label = "Safe?"

data = [
    ["round", "large", "blue", "no"],
    ["square", "large", "green", "yes"],
    ["square", "small", "red", "no"],
    ["round", "large", "red", "yes"],
    ["square", "small", "blue", "no"],
    ["round", "small", "blue", "no"],
    ["round", "small", "red", "yes"],
    ["square", "small", "green", "no"],
    ["round", "large", "green", "yes"],
    ["square", "large", "green", "yes"],
    ["square", "large", "red", "no"],
    ["square", "large", "green", "yes"],
    ["round", "large", "red", "yes"],
    ["square", "small", "red", "no"],
    ["round", "small", "green", "no"],
]

run_model(data, attributes, label)


Confusion Matrix (10-fold CV):
TP=6  FP=1
FN=2  TN=6

Average Error Rate (10-fold CV): 0.2000

Decision Tree:
 - Size
 | large
         - Color
         | blue -> no
         | green -> yes
         | red
                 - Shape
                 | round -> yes
                 | square -> no
 | small
         - Shape
         | round
                 - Color
                 | blue -> no
                 | green -> no
                 | red -> yes
         | square -> no


In [ ]:
trace = False
data = parse_data("./agaricus-lepiota-3.data")
attributes, abrv2fullname = parse_attributes("./agaricus-lepiota-3.attrs")
label = "mushroom-type"
data = rename_data(data, attributes, abrv2fullname)
assert data is not None

run_model(data, attributes, label)

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.